In [1]:
import pandas as pd
import openai
import json
import backoff
from tqdm import tqdm
import numpy as np
import os

## Load Procedures and Teaching Tasks

In [2]:
procedures_df = pd.read_excel('/home/firdavs/surgery/annotations/Overview of Cases.xlsx')
procedures_df = procedures_df.rename(columns={'LFB Case #': 'case_id', 'Procedure': 'procedure'})
procedures_df = procedures_df[['case_id', 'procedure']]
procedures_df.head()

,case_id,procedure
0,1,radical prostatectomy
1,2,simple prostatectomy
2,3,radical prostatectomy
3,4,radical prostatectomy
4,5,radical prostatectomy


In [5]:
teaching_steps_df = pd.read_csv('/home/firdavs/surgery/annotations/console_times/combined_console_times_secs.csv')
teaching_steps_df.reset_index(drop=True, inplace=True)
teaching_steps_df = teaching_steps_df.rename(columns={'Teaching Step': 'teaching_step', 'On time (secs)': 'start_secs', 'Off time (secs)': 'end_secs'})
teaching_steps_df = teaching_steps_df[['case_id', 'teaching_step', 'start_secs', 'end_secs']]
teaching_steps_df.head()

,case_id,teaching_step,start_secs,end_secs
0,1,Lymph node dissection,-28,1802
1,1,Dropping bladder,1802,2042
2,1,Endopelvic fascia,2042,2132
3,1,Bladder neck,2132,2762
4,1,Bladder neck,2762,2912


In [11]:
procedures_df['procedure'].unique()

array(['radical prostatectomy', 'simple prostatectomy', 'nephrectomy',
       'nephroureterectomy', 'partial nephrectomy',
       'inguinal hernia repair', 'cystectomy and ilial conduit '],
      dtype=object)

In [24]:
teaching_steps_df['teaching_step'].unique()

array(['Lymph node dissection', 'Dropping bladder ', 'Endopelvic fascia',
       'Bladder neck', 'SV', 'posterior plane', 'Posterior plane',
       'Bladder neck recon', 'Pedicle/NVB', 'apical dissection', 'DVC',
       'oblique checks', 'hemostasis, rocco stitch', 'VUA',
       'Adenoma dissection', 'Dropping bladder', nan, 'Bladder Neck',
       'Pedicles/NVB/apical dissection', 'Hemostasis', 'Rocco stitch',
       'dropping bladder', 'Pedicles/NVB', 'Apical dissection',
       'DVC, hemostasis', 'Rocco', 'Pexy',
       'Bowel mobilization + Lymph dissection', 'Endopelvic fascia ',
       'Bladder neck / SV', 'pedicles / NVB', 'apical dissection ',
       'Hemostasis, DVC', 'pexy', 'extra node dissection',
       'Bowel mobilization', 'dissection', 'dissecting ',
       'stapling ureter ', 'Bagging specimen ',
       'undocking, specimen out, closing incisions', 'docking robot',
       '(setting up, waiting for attending)', ' Cystotomy',
       'Stay Sutures', 'Adenoma Dissection', '

## Generate Definitions

In [13]:
# Get OpenAI API key
with open('../keys.json', 'r') as f:
    keys = json.load(f)
openai_api_key = keys['openai_api_key-personal']

In [14]:
#@title Function for Initializing OpenAI and Prompting
def initOpenAI(key):
    client = openai.OpenAI(
    # This is the default and can be omitted
    api_key=key,
    )
    return client

# Function to List Available Models
def listModels(client):
    models = client.models.list()
    model_names = [model.id for model in models.data]
    return model_names

In [25]:
def get_prompt_procedure(procedure):
    instruction = f"You are working in the domain of urology surgery. You are tasked with defining surgical procedures in 3-4 sentences."
    label_prompt = f"Define the surgical procedure: {procedure}"

    return instruction, label_prompt

def get_prompt_teaching_step(teaching_step):
    instruction = f"You are working in the domain of urology surgery. You are tasked with defining surgical teaching steps in 1-2 sentences."
    label_prompt = f"Define the teaching step: {teaching_step}"

    return instruction, label_prompt
    
def labelGPT(client, model_name, instruction, label_prompt, temperature=0.0):
    @backoff.on_exception(backoff.expo, (openai.RateLimitError,
                                        openai.APIError,
                                        ConnectionResetError,
                                        json.decoder.JSONDecodeError))

    def completions_with_backoff(**kwargs):
        return client.chat.completions.create(**kwargs)

    # Prompt OpenAI
    # https://github.com/openai/openai-python
    response = completions_with_backoff(model=model_name,
                                        temperature=temperature,
                                        messages=[{"role": "system", "content": instruction},
                                                {"role":"user", "content": label_prompt}])

    gen = response.choices[0].message.content
    return gen

In [16]:
openai_client = initOpenAI(openai_api_key)
available_models = listModels(openai_client)
model_name = 'gpt-4o-mini'

In [17]:
procedure_mappings = {}
unique_procedures = procedures_df['procedure'].unique().tolist()
for i in tqdm(range(len(unique_procedures)), desc='Procedures'):
    procedure = unique_procedures[i]
    instruction, label_prompt = get_prompt_procedure(procedure)
    gen = labelGPT(openai_client, model_name, instruction, label_prompt)
    procedure_mappings[procedure] = gen

Procedures: 100%|██████████| 7/7 [00:23<00:00,  3.30s/it]


In [18]:
print(json.dumps(procedure_mappings, indent=4))

{
    "radical prostatectomy": "Radical prostatectomy is a surgical procedure that involves the complete removal of the prostate gland along with some surrounding tissue, including the seminal vesicles and sometimes nearby lymph nodes. This procedure is primarily performed to treat localized prostate cancer and aims to eliminate cancerous cells while preserving as much surrounding healthy tissue as possible. It can be done through open surgery or minimally invasive techniques, such as laparoscopic or robotic-assisted surgery. Postoperative recovery may involve managing urinary incontinence and erectile dysfunction, which are common side effects.",
    "simple prostatectomy": "A simple prostatectomy is a surgical procedure aimed at removing the prostate gland and some surrounding tissue to alleviate symptoms caused by benign prostatic hyperplasia (BPH) or prostate cancer. This procedure is typically performed through an incision in the lower abdomen or via a transurethral approach, depe

In [35]:
procedures_df['procedure_defn'] = procedures_df['procedure'].map(procedure_mappings)
procedures_df.to_parquet('/home/firdavs/surgery/surgical_fb_generation/SurgicalFeedbackGeneration/data/urology-related/procedures.parquet', index=False)

In [27]:
teaching_step_mapppings = {}
unique_teaching_steps = teaching_steps_df['teaching_step'].unique().tolist()
for i in tqdm(range(len(unique_teaching_steps)), desc='Teaching Steps'):
    teaching_step = unique_teaching_steps[i]
    instruction, label_prompt = get_prompt_teaching_step(teaching_step)
    gen = labelGPT(openai_client, model_name, instruction, label_prompt)
    teaching_step_mapppings[teaching_step] = gen

Teaching Steps: 100%|██████████| 118/118 [02:52<00:00,  1.46s/it]


In [28]:
print(json.dumps(teaching_step_mapppings, indent=4))

{
    "Lymph node dissection": "Lymph node dissection involves the surgical removal of lymph nodes to assess for cancer spread, typically performed in conjunction with other procedures such as radical prostatectomy or cystectomy. The teaching step includes identifying the anatomical landmarks, ensuring proper technique to minimize complications, and understanding the implications of lymphatic drainage in urological malignancies.",
    "Dropping bladder ": "Dropping the bladder involves carefully mobilizing the bladder from its anatomical attachments to the pelvic structures, ensuring minimal trauma to surrounding tissues, and providing clear visualization of the surgical field for subsequent procedures. This step is crucial for accessing the prostate or other pelvic organs while maintaining hemostasis and preserving bladder function.",
    "Endopelvic fascia": "The endopelvic fascia is a connective tissue layer that supports pelvic organs and is crucial in urological surgeries. During 

In [34]:
teaching_steps_df['teaching_step_defn'] = teaching_steps_df['teaching_step'].map(teaching_step_mapppings)
teaching_steps_df.to_parquet('/home/firdavs/surgery/surgical_fb_generation/SurgicalFeedbackGeneration/data/urology-related/teaching_steps.parquet', index=False)

## Extract Embeddings

In [83]:
from sentence_transformers import SentenceTransformer

def get_medembed_embeddings(texts):
    medembed = SentenceTransformer('abhinand/MedEmbed-small-v0.1')
    embs = []
    for i in tqdm(range(len(texts)), desc='MedEmbed Embedding'):
        text = texts[i]
        emb = medembed.encode(text)
        embs.append(emb)
    return embs

In [74]:
from mmengine.config import Config
SurgVLP_path = '/home/firdavs/surgery/surgical_fb_generation/SurgVLP'
import sys
sys.path.append(SurgVLP_path)
import surgvlp

def get_vl_embs(model_name, texts, device='cuda'):    # model_name = 'surgvlp', 'peskavlp', 'hecvl'
    configs = Config.fromfile(os.path.join(SurgVLP_path, 'tests', 'config_peskavlp.py'), lazy_import=False)['config']
    model, _ = surgvlp.load(configs.model_config, device=device, strict_load_state_dict=False)
    text_embs = []
    for i in tqdm(range(len(texts)), desc=f'{model_name} Embedding'):
        text = surgvlp.tokenize([texts[i]], device=device)
        output_dict = model(None, text , mode='text')
        text_embs.append(output_dict['text_emb'][0].cpu().detach().numpy())
    return text_embs

### 1) Procedures

In [60]:
procedure_defn_embeddings = {
    'MedEmbed_small': [],
    'SurgVLP': [],
    'HecVL': [],
    'PeskaVLP': [],
}

In [ ]:
procedure_defn_embeddings['MedEmbed_small'] = get_medembed_embeddings(procedures_df['procedure_defn'].tolist())

MedEmbed Procedures Embedding: 100%|██████████| 35/35 [00:00<00:00, 251.69it/s]


In [62]:
procedure_defn_embeddings['SurgVLP'] = get_vl_embs('surgvlp', procedures_df['procedure_defn'].tolist())
procedure_defn_embeddings['HecVL'] = get_vl_embs('hecvl', procedures_df['procedure_defn'].tolist())
procedure_defn_embeddings['PeskaVLP'] = get_vl_embs('peskavlp', procedures_df['procedure_defn'].tolist())

peskavlp Embedding: 100%|██████████| 35/35 [00:27<00:00,  1.27it/s]


In [68]:
for k, v in procedure_defn_embeddings.items():
    procedures_df[f'procedure_defn_emb-{k}'] = v

In [72]:
procedures_df.to_parquet('/home/firdavs/surgery/surgical_fb_generation/SurgicalFeedbackGeneration/data/urology-related/procedures.parquet', index=False)

### 2) Teaching Steps

In [76]:
teaching_step_defn_embeddings = {
    'MedEmbed_small': [],
    'SurgVLP': [],
    'HecVL': [],
    'PeskaVLP': [],
}

In [84]:
teaching_step_defn_embeddings['MedEmbed_small'] = get_medembed_embeddings(teaching_steps_df['teaching_step_defn'].tolist())

MedEmbed Embedding: 100%|██████████| 489/489 [00:01<00:00, 309.48it/s]


In [78]:
teaching_step_defn_embeddings['SurgVLP'] = get_vl_embs('surgvlp', teaching_steps_df['teaching_step_defn'].tolist())
teaching_step_defn_embeddings['HecVL'] = get_vl_embs('hecvl', teaching_steps_df['teaching_step_defn'].tolist())
teaching_step_defn_embeddings['PeskaVLP'] = get_vl_embs('peskavlp', teaching_steps_df['teaching_step_defn'].tolist())

/home/firdavs/miniconda3/envs/firdavs/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/firdavs/miniconda3/envs/firdavs/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
peskavlp Embedding: 100%|██████████| 489/489 [06:47<00:00,  1.20it/s]


In [86]:
for k, v in teaching_step_defn_embeddings.items():
    teaching_steps_df[f'teaching_step_defn_emb-{k}'] = v

In [88]:
teaching_steps_df.to_parquet('/home/firdavs/surgery/surgical_fb_generation/SurgicalFeedbackGeneration/data/urology-related/teaching_steps.parquet', index=False)